# 26 -- Striatum-crop-coverage fix: recentered-constant vs. per-subject-centered CNN (2026-09-13)

Phase B of the 5th Opus review's plan (see project memory
`project_dat_parkinson_strategic_roadmap.md`). Phase A (notebook 25's
op08-op11) confirmed: the crop-window bug found in op04 barely moved
coverage (70.2%->71.1%); the family/FOV confound is real but partial (the
CNN's within-family coverage gap, n=528, single family, is still large and
clearly significant: +0.1758, 95% CI=[+0.0911,+0.2617], while the baseline's
is small and crosses 0); the AUROC decomposition is the cleanest evidence
yet that this is a CNN discrimination failure, not general subject
difficulty (covered AUROC=0.9293 -> uncovered AUROC=0.7088, vs. the
baseline's 0.8112 -> 0.7666); and no blend-covariate fix (binary or
continuous, at n_bootstrap=1000 or 20000) recovers more than a sliver of
it -- exactly what "the CNN never saw the anatomy" predicts, since a blend
covariate can only correct calibration, not discrimination.

Hypothetical coverage (op08): current 71.1%; re-centering the fixed
`CROP_CENTER_MM` constant to op03's corrected median (2.3, 22.4, -38.1mm)
-> 79.0%; per-subject centering -> 100.0% by construction.

This notebook trains a rung3-equivalent CNN variant (same architecture,
same 5-seed x 5-fold nested-CV protocol as `notebooks/07_cnn_rung3.ipynb`,
nothing else changed) under each of the two crop-center arms, in separate
on-disk caches (never touching the shared `volume_cache` all 6 production
variants depend on), and gates both against `rung3_oof_seed{42-46}.npy`
via `evaluate.paired_repeat_gate` -- a dose-response check (79% vs. 100%
coverage) is more persuasive than either arm alone, and per the review,
per-subject centering is NOT more expensive than the "cheap" global
re-center once the per-uid centroids already exist on disk
(`data/processed/nb25_op02audit_checkpoint.npz`, built by
`notebooks/25_striatum_centroid_ras_frame_audit.ipynb`'s op02audit cell).

Decision deadline (per the review): end of 2026-09-14. If neither arm
clears convincingly by then, ship nothing further and keep 0.4185.

In [ ]:
# [RUN ME] -- loads real row-level labels + the per-uid striatum centroids
# already measured and checkpointed by notebooks/25's op02audit cell.
# Self-contained: does not assume notebook 25 is still warm in this kernel.
import functools
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import torch

import cache
import config
import data
import dataset
import evaluate
import model
import train as train_mod

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)

uids = labeled_df[config.UID_COLUMN].tolist()
labels = labeled_df[config.TARGET_COLUMN].tolist()
families = labeled_df["inplane_family"].tolist()

checkpoint_path = config.DATA_PROCESSED / "nb25_op02audit_checkpoint.npz"
ckpt = np.load(checkpoint_path, allow_pickle=True)
ckpt_uids = list(ckpt["uids"])
assert int(ckpt["next_i"]) == len(ckpt_uids) and int(ckpt["n_degenerate"]) == 0, (
    "nb25_op02audit_checkpoint.npz is incomplete or has degenerate volumes -- "
    "re-run notebooks/25's op02audit cell to completion before trusting the "
    "per-uid offsets below (this notebook assumes exactly one offset row per uid, "
    "in labels_df order, per the confirmed n_degenerate=0 result)."
)
assert ckpt_uids == uids, (
    "checkpoint's uid order/set doesn't match labels_df -- the offsets would "
    "misalign with labels/families below."
)
uid_to_offset = dict(zip(ckpt_uids, ckpt["offsets_mm"]))

RECENTERED_OFFSET = (2.3, 22.4, -38.1)  # op03's corrected median (x, y, z), notebooks/25

print(f"{len(uids)} labeled rows, {len(uid_to_offset)} per-uid centroid offsets loaded "
      f"(recentered arm target: {RECENTERED_OFFSET}).")


In [ ]:
# [RUN ME] (no data access itself). Identical protocol to
# notebooks/07_cnn_rung3.ipynb's train_and_score_nested (early stopping on
# a single 90/10 inner split, batch_size=32, lr=2e-3) -- the ONLY thing
# that changes between this experiment and rung3 is which volume_cache
# feeds the dataset. `volume_cache` is now an explicit parameter (not a
# captured global) so the same helper serves both arms below.
def train_and_score_nested(volume_cache, train_uids, train_labels, train_family,
                            outer_uids, batch_size, lr, seed,
                            epochs=config.EPOCHS, patience=config.PATIENCE,
                            inner_splits=10):
    inner_train_idx, inner_val_idx = evaluate.make_folds(
        train_labels, train_family, n_splits=inner_splits, random_state=seed
    )[0]

    def subset(idxs):
        return ([train_uids[i] for i in idxs], [train_labels[i] for i in idxs])

    inner_train_uids, inner_train_labels = subset(inner_train_idx)
    inner_val_uids, inner_val_labels = subset(inner_val_idx)

    inner_train_ds = dataset.DatParkinsonDataset(inner_train_uids, inner_train_labels, load_fn=volume_cache.get)
    inner_val_ds = dataset.DatParkinsonDataset(inner_val_uids, inner_val_labels, load_fn=volume_cache.get)
    inner_train_loader = torch.utils.data.DataLoader(inner_train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    inner_val_loader = torch.utils.data.DataLoader(inner_val_ds, batch_size=batch_size, num_workers=0)

    torch.manual_seed(seed)
    net = model.build_model().to(config.DEVICE)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=config.WEIGHT_DECAY)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    best_state, history = train_mod.train_one_fold(
        net, inner_train_loader, inner_val_loader, optimizer, loss_fn,
        epochs=epochs, patience=patience, device=config.DEVICE,
        use_amp=config.USE_AMP, seed=seed,
    )
    net.load_state_dict(best_state)

    outer_ds = dataset.DatParkinsonDataset(outer_uids, load_fn=volume_cache.get)
    outer_loader = torch.utils.data.DataLoader(outer_ds, batch_size=batch_size, num_workers=0)
    outer_probs = []
    for x, _ in outer_loader:
        outer_probs.append(model.predict(net, x))
    return np.concatenate(outer_probs), history, best_state


def run_nested_cv(volume_cache, oof_prefix, checkpoint_prefix, n_repeats=5, batch_size=32, lr=2e-3):
    """Full 5-fold nested CV, repeated n_repeats times -- identical outer-loop
    structure to notebooks/07/09-12/18. Saves `{oof_prefix}_oof_seed{s}.npy`
    and `{checkpoint_prefix}_seed{s}_fold{i}.pt`, returns the list of
    per-repeat OOF arrays."""
    y_true = np.array(labels)
    oof_repeats = []
    for repeat_seed in range(config.SEED, config.SEED + n_repeats):
        outer_folds = evaluate.make_folds(y_true, np.array(families),
                                           n_splits=config.N_FOLDS, random_state=repeat_seed)
        oof_probs = np.zeros(len(uids))
        for fold_i, (train_idx, test_idx) in enumerate(outer_folds):
            fold_train_uids = [uids[i] for i in train_idx]
            fold_train_labels = [labels[i] for i in train_idx]
            fold_train_family = [families[i] for i in train_idx]
            fold_test_uids = [uids[i] for i in test_idx]

            fold_start = time.time()
            probs, history, best_state = train_and_score_nested(
                volume_cache, fold_train_uids, fold_train_labels, fold_train_family,
                fold_test_uids, batch_size=batch_size, lr=lr, seed=repeat_seed,
            )
            oof_probs[test_idx] = probs
            torch.save(best_state, config.CHECKPOINT_DIR / f"{checkpoint_prefix}_seed{repeat_seed}_fold{fold_i}.pt")
            fold_score = evaluate.log_loss_score(y_true[test_idx], probs)
            print(f"  seed={repeat_seed} fold={fold_i}: {len(history['val_loss'])} epochs, "
                  f"outer fold log loss={fold_score:.4f}, {time.time() - fold_start:.1f}s")

        repeat_logloss = evaluate.log_loss_score(y_true, oof_probs)
        oof_repeats.append(oof_probs)
        print(f"seed={repeat_seed} pooled OOF log loss: {repeat_logloss:.4f}")
        np.save(config.DATA_PROCESSED / f"{oof_prefix}_oof_seed{repeat_seed}.npy", oof_probs)
    return oof_repeats


In [ ]:
# [RUN ME] -- builds the RECENTERED-CONSTANT cache: same CNN input pipeline
# as production, only CROP_CENTER_MM changed to op03's corrected median
# (2.3, 22.4, -38.1). Separate cache_dir (never touches the shared
# volume_cache all 6 production variants depend on). CPU-only, same order
# of cost as the shared volume_cache's own original build (resample+crop+
# normalize per volume, no mask/feature extraction).
recenter_fingerprint = {
    "TARGET_SPACING": config.TARGET_SPACING,
    "CENTER_MODE": "recentered_constant",
    "CENTER_MM": RECENTERED_OFFSET,
    "TARGET_SHAPE": config.TARGET_SHAPE,
    "BACKGROUND_PERCENTILE": config.BACKGROUND_PERCENTILE,
    "BACKGROUND_MAX_FRACTION": config.BACKGROUND_MAX_FRACTION,
}
recenter_load_fn = functools.partial(data.load_volume, center_mm=RECENTERED_OFFSET)

cache_start = time.time()
recenter_cache = cache.CachedVolumeStore(
    uids, cache_dir=config.DATA_PROCESSED / "volume_cache_recentered",
    config_fingerprint=recenter_fingerprint, load_fn=recenter_load_fn,
)
print(f"recentered-constant cache {'reused' if recenter_cache.was_reused else 'rebuilt'} "
      f"in {time.time() - cache_start:.1f}s for {len(uids)} volumes")


In [ ]:
# [RUN ME] -- full 5-fold nested CV, repeated 5x, on the RECENTERED-CONSTANT
# arm. Same outer-loop protocol as rung 3.
N_REPEATS = 5
oof_repeats_recenter = run_nested_cv(
    recenter_cache, oof_prefix="coveragefix_recenter", checkpoint_prefix="coveragefix_recenter",
    n_repeats=N_REPEATS,
)
repeat_scores_recenter = np.array([evaluate.log_loss_score(np.array(labels), oof) for oof in oof_repeats_recenter])
print(f"\n{N_REPEATS}-repeat recentered-constant CNN pooled log loss: "
      f"mean={repeat_scores_recenter.mean():.4f}, sd={repeat_scores_recenter.std(ddof=1):.4f}")
print("for reference: rung3 (current production geometry) = 0.4520")


In [ ]:
# [RUN ME] -- builds the PER-SUBJECT-CENTERED cache: each uid crops around
# its OWN measured striatum-centroid offset (uid_to_offset, from
# notebooks/25's op02audit) instead of any single fixed constant -- the
# "full fix", 100% coverage by construction (op08). Separate cache_dir,
# never touches the shared volume_cache or the recentered-constant cache
# above. CPU-only, same order of cost as building the recentered cache.
def _persubject_load_fn(uid):
    return data.load_volume(uid, center_mm=tuple(uid_to_offset[uid]))


persubject_fingerprint = {
    "TARGET_SPACING": config.TARGET_SPACING,
    "CENTER_MODE": "per_subject",
    "CENTER_SOURCE": "nb25_op02audit_checkpoint.npz",
    "TARGET_SHAPE": config.TARGET_SHAPE,
    "BACKGROUND_PERCENTILE": config.BACKGROUND_PERCENTILE,
    "BACKGROUND_MAX_FRACTION": config.BACKGROUND_MAX_FRACTION,
}

cache_start = time.time()
persubject_cache = cache.CachedVolumeStore(
    uids, cache_dir=config.DATA_PROCESSED / "volume_cache_persubject",
    config_fingerprint=persubject_fingerprint, load_fn=_persubject_load_fn,
)
print(f"per-subject cache {'reused' if persubject_cache.was_reused else 'rebuilt'} "
      f"in {time.time() - cache_start:.1f}s for {len(uids)} volumes")


In [ ]:
# [RUN ME] -- full 5-fold nested CV, repeated 5x, on the PER-SUBJECT arm.
# Same outer-loop protocol as rung 3 and the recentered-constant arm above
# -- only the cache (and therefore each volume's crop center) differs.
oof_repeats_persubject = run_nested_cv(
    persubject_cache, oof_prefix="coveragefix_persubject", checkpoint_prefix="coveragefix_persubject",
    n_repeats=N_REPEATS,
)
repeat_scores_persubject = np.array([evaluate.log_loss_score(np.array(labels), oof) for oof in oof_repeats_persubject])
print(f"\n{N_REPEATS}-repeat per-subject-centered CNN pooled log loss: "
      f"mean={repeat_scores_persubject.mean():.4f}, sd={repeat_scores_persubject.std(ddof=1):.4f}")
print("for reference: rung3 (current production geometry) = 0.4520, "
      f"recentered-constant arm = {repeat_scores_recenter.mean():.4f}")


In [ ]:
# [RUN ME] (no data access itself). Gate both arms against the current
# validated CNN using evaluate.paired_repeat_gate (paired per-repeat delta,
# t-CI on sd/sqrt(n) -- same statistic used to close out rung 4, see
# project_dat_parkinson_rung4_gate_review.md). A dose-response pattern
# (79% coverage arm beats rung3 by less than the 100% coverage arm, both
# in the same direction) is stronger evidence than either gate alone.
y_true = np.array(labels)
repeat_seeds = list(range(config.SEED, config.SEED + N_REPEATS))
current_cnn_oof_by_repeat = [np.load(config.DATA_PROCESSED / f"rung3_oof_seed{s}.npy") for s in repeat_seeds]

for label, oof_repeats in [
    ("recentered-constant (79% coverage)", oof_repeats_recenter),
    ("per-subject-centered (100% coverage)", oof_repeats_persubject),
]:
    deltas = [
        evaluate.log_loss_score(y_true, oof_repeats[i]) - evaluate.log_loss_score(y_true, current_cnn_oof_by_repeat[i])
        for i in range(N_REPEATS)
    ]
    gate = evaluate.paired_repeat_gate(deltas)
    print(f"{label}:")
    print(f"  per-repeat deltas (candidate - rung3): {[f'{d:+.4f}' for d in deltas]}")
    print(f"  mean={gate['mean']:+.4f}, sd={gate['sd']:.4f}, 95% CI=[{gate['ci_low']:+.4f}, {gate['ci_high']:+.4f}]")
    print(f"  GATE {'PASSED' if gate['passed'] else 'NOT PASSED'}: "
          f"{'REPLACES rung3.' if gate['passed'] else 'does not beat rung3 by more than noise.'}\n")

print("dose-response check: if both means are negative and the per-subject arm's |mean| "
      "is clearly larger than the recentered-constant arm's, that's the coverage-fix "
      "mechanism confirmed operating at the CNN-training level (not just the blend), "
      "scaling with how much of the coverage gap each arm actually closes (79% vs 100%).")


**What we're looking for:** does actually retraining with a corrected
crop center recover some of the CNN discrimination loss Phase A measured
(covered AUROC 0.9293 vs. uncovered 0.7088) -- something no blend
covariate could do, since a blend can only recalibrate, never add
information the CNN never saw? Two arms: a cheap global re-center (79%
coverage) and the full per-subject fix (100% coverage), gated separately
against `rung3_oof_seed{42-46}.npy` via `evaluate.paired_repeat_gate`
(the same statistic that closed out rung 4).

**What we found:**

*(fill in after running op03-op07 -- print output above reports both
arms' per-repeat pooled log loss, the gate verdict for each vs. rung3, and
the dose-response comparison)*

**Decision / next step:** per the 5th Opus review's plan, this is Phase B.
If either arm's gate passes (or both, with a clear dose-response pattern),
proceed to Phase C: retrain all 6 production variants on whichever cache
composition wins, re-run notebook 19/22-style calibration+blend
re-tuning, repackage, local Docker + platform smoke tests, then decide on
the last submission -- budget a full day for the `submission_src/main.py`
inference-side change (computing each test volume's own centroid, if the
per-subject arm wins) plus both smoke tests. If neither arm clears,
log this as a real, honestly-tested negative result (same treatment as
denoise/flip-TTA/slab2d) and close the crop-coverage investigation at
0.4185. Decision deadline: end of 2026-09-14.